In [ ]:
import os
import ipynbname

notebook_name = ipynbname.name() + ".ipynb"

os.system(f'jupyter nbconvert "{notebook_name}" --to markdown')

In [1]:
import os
import re
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import LongformerTokenizer, LongformerModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef, f1_score, classification_report, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ==========================================
# 0. REPRODUCIBILITY & KAGGLE CONFIGURATION
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

SMELL_TYPE = 'blob'

# --- KAGGLE PATHS ---
DATASET_ROOT = "/kaggle/input/datasets/sahelsoft"
PMD_CSV_PATH = f"{DATASET_ROOT}/pmdextractedfeatures/PMDExtractedFeatures.csv"
MLCQ_CSV_PATH = f"{DATASET_ROOT}/mlcqcodesmellsamples/MLCQCodeSmellSamples.csv"
CODE_DIR = f"{DATASET_ROOT}/mlcq_dataset" 

OUTPUT_DIR = '/kaggle/working/Outputs'
FIG_DIR = '/kaggle/working/Paper_Figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_SAVE_PATH = f'{OUTPUT_DIR}/{SMELL_TYPE}_sota_ds1_binary.pth'

BATCH_SIZE = 1           
ACCUMULATION_STEPS = 32  # Effective batch size = 32
EPOCHS = 15
MAX_LEN = 4096           

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# PHASE 1: SOTA DATA PREPARATION (DS1 BINARY)
# ==========================================
print("\n--- Phase 1: Data Preparation ---")

print("Loading PMD Features...")
pmd_df = pd.read_csv(PMD_CSV_PATH)
pmd_df['sample_id'] = pmd_df['ID'].str.replace('.java', '', regex=False).astype(int)
pmd_df = pmd_df.drop(columns=['ID']).set_index('sample_id').fillna(0)

# SOTA FIX 1: Log-Transform skewed PMD metrics to fix Neural Network saturation
pmd_df = np.log1p(np.abs(pmd_df))
METRICS_DIM = pmd_df.shape[1]

print("Loading MLCQ Labels & Generating DS1 Binary Targets...")
mlcq_df = pd.read_csv(MLCQ_CSV_PATH)
target_smell_df = mlcq_df[mlcq_df['smell'] == SMELL_TYPE].copy()

# Cross-tabulate votes
vote_counts = pd.crosstab(target_smell_df['sample_id'], target_smell_df['severity'])
for sev in ['none', 'minor', 'major', 'critical']:
    if sev not in vote_counts.columns: vote_counts[sev] = 0

# Convert directly to DS1 Binary: Smelly (minor+major+critical) vs Clean (none)
ds1_smelly_votes = vote_counts['minor'] + vote_counts['major'] + vote_counts['critical']
ds1_clean_votes = vote_counts['none']

# Majority vote determines the binary ground truth
binary_labels = (ds1_smelly_votes > ds1_clean_votes).astype(int)
binary_labels.name = 'label'

valid_sample_ids = binary_labels.index.intersection(pmd_df.index).tolist()
binary_labels = binary_labels.loc[valid_sample_ids]

# SOTA FIX 2: Calculate Positive Weight for BCEWithLogitsLoss
num_negatives = (binary_labels == 0).sum()
num_positives = (binary_labels == 1).sum()
pos_weight_val = num_negatives / max(1, num_positives)
pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
print(f"DS1 Class Distribution -> Clean: {num_negatives}, Smelly: {num_positives}")
print(f"BCE Positive Weight applied: {pos_weight_val:.2f}")

# Stratified Splitting for DS1
train_ids, temp_ids = train_test_split(valid_sample_ids, test_size=0.2, random_state=42, stratify=binary_labels)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42, stratify=binary_labels.loc[temp_ids])

# Leakage-Free Scaling
scaler = StandardScaler()
pmd_df.loc[train_ids] = scaler.fit_transform(pmd_df.loc[train_ids])
pmd_df.loc[val_ids] = scaler.transform(pmd_df.loc[val_ids])
pmd_df.loc[test_ids] = scaler.transform(pmd_df.loc[test_ids])

def clean_java_code(code):
    code = re.sub(r'/\*[\s\S]*?\*/', '', code)     
    code = re.sub(r'(?<!:)//.*', '', code)         
    code = re.sub(r'import\s+[\w\.]+;', '', code)  
    code = re.sub(r'package\s+[\w\.]+;', '', code) 
    return os.linesep.join([s for s in code.splitlines() if s.strip()]) 

class BinaryMultiModalDataset(Dataset):
    def __init__(self, sample_ids, pmd_df, labels, code_dir, tokenizer):
        self.sample_ids = sample_ids
        self.pmd_df = pmd_df
        self.labels = labels
        self.code_dir = code_dir
        self.tokenizer = tokenizer 

    def __len__(self): return len(self.sample_ids)

    def __getitem__(self, idx):
        s_id = self.sample_ids[idx]
        metrics = torch.tensor(self.pmd_df.loc[s_id].values, dtype=torch.float32)
        # Shape must be [1] for BCEWithLogitsLoss
        label = torch.tensor([self.labels.loc[s_id]], dtype=torch.float32)
        
        file_path = os.path.join(self.code_dir, f"{s_id}.java")
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                code_content = clean_java_code(f.read())
        except:
            code_content = ""

        encoding = self.tokenizer(
            code_content, add_special_tokens=True, max_length=MAX_LEN,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'sample_id': s_id,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'metrics': metrics,
            'label': label
        }

print("Initializing Tokenizer...")
tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")

train_dataset = BinaryMultiModalDataset(train_ids, pmd_df, binary_labels, CODE_DIR, tokenizer)
val_dataset   = BinaryMultiModalDataset(val_ids,   pmd_df, binary_labels, CODE_DIR, tokenizer)
test_dataset  = BinaryMultiModalDataset(test_ids,  pmd_df, binary_labels, CODE_DIR, tokenizer)

# num_workers=0 to prevent Kaggle DataLoader serialization crash
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# ==========================================
# PHASE 2: WIDE & DEEP GATED LONGFORMER (BINARY)
# ==========================================
class BinaryGatedLongformer(nn.Module):
    def __init__(self, metrics_input_dim, hidden_dim=768, dropout_prob=0.3): 
        super().__init__()
        self.code_encoder = LongformerModel.from_pretrained("allenai/longformer-base-4096")
        
        self.metrics_encoder = nn.Sequential(
            nn.Linear(metrics_input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(512, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        
        self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True, dropout=dropout_prob)
        
        self.gate_network = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )
        
        # SOTA FIX 3: Binary output head (1 unit)
        self.binary_head = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim + metrics_input_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(256, 1) # Single logit for BCE
        )
        
    def forward(self, input_ids, attention_mask, metrics):
        global_attention_mask = torch.zeros_like(input_ids)
        global_attention_mask[:, 0] = 1 
        
        code_outputs = self.code_encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        code_seq = code_outputs.last_hidden_state 
        metrics_emb = self.metrics_encoder(metrics).unsqueeze(1) 
        
        key_padding_mask = (attention_mask == 0)
        attn_output, attn_weights = self.cross_attention(
            query=metrics_emb, key=code_seq, value=code_seq, key_padding_mask=key_padding_mask
        )
        context_vector = attn_output.squeeze(1)
        metrics_flat = metrics_emb.squeeze(1)
        
        gate_val = self.gate_network(torch.cat([context_vector, metrics_flat], dim=1))
        fused_vector = (gate_val * context_vector) + ((1 - gate_val) * metrics_flat)
        
        # Wide & Deep Residual Connection
        final_representation = torch.cat([fused_vector, metrics], dim=1)
        
        logits = self.binary_head(final_representation)
        return logits, attn_weights.squeeze(1)

print("\nInitializing Wide & Deep Binary Longformer Model...")
model = BinaryGatedLongformer(metrics_input_dim=METRICS_DIM).to(device)

# ==========================================
# PHASE 3: SOTA TRAINING & F1 OPTIMIZATION
# ==========================================
print("\n--- Phase 3: Training with Differential Learning Rates ---")

# SOTA FIX 4: Differential Learning Rates!
optimizer_grouped_parameters = [
    {'params': model.code_encoder.parameters(), 'lr': 2e-5},
    {'params': model.metrics_encoder.parameters(), 'lr': 1e-3},
    {'params': model.cross_attention.parameters(), 'lr': 1e-3},
    {'params': model.gate_network.parameters(), 'lr': 1e-3},
    {'params': model.binary_head.parameters(), 'lr': 1e-3}
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=0.05)
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

# Loss function specifically designed for Imbalanced Binary tasks
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

best_val_f1 = -1.0
optimal_ds1_threshold = 0.5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)):
        logits, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['metrics'].to(device))
        
        loss = criterion(logits, batch['label'].to(device))
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * ACCUMULATION_STEPS

    # VALIDATION
    model.eval()
    val_preds_prob, val_trues_bin = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['metrics'].to(device))
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            labels = batch['label'].cpu().numpy().flatten()
            
            val_preds_prob.extend(probs)
            val_trues_bin.extend(labels)
            
    # Tuning Threshold specifically for F1-SCORE (SOTA Fix 5)
    best_thresh, best_f1, best_mcc_at_f1 = 0.5, -1.0, -1.0
    for thresh in np.arange(0.1, 0.9, 0.05):
        bin_preds = (np.array(val_preds_prob) > thresh).astype(int)
        f1 = f1_score(val_trues_bin, bin_preds, zero_division=0)
        mcc = matthews_corrcoef(val_trues_bin, bin_preds)
        
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
            best_mcc_at_f1 = mcc
            
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {best_f1:.4f} | Val MCC: {best_mcc_at_f1:.4f} (at Threshold {best_thresh:.2f})")
    
    if best_f1 > best_val_f1:
        best_val_f1 = best_f1
        optimal_ds1_threshold = best_thresh
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("   -> 🌟 New Best F1 Model Saved!")

# ==========================================
# PHASE 4: FINAL TEST EVALUATION
# ==========================================
print(f"\n--- Phase 4: Final Testing ---")
print(f"Applying Dynamically Tuned F1 Threshold: {optimal_ds1_threshold:.2f}")

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

preds_ds1_prob, trues_ds1 = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        logits, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['metrics'].to(device))
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        labels = batch['label'].cpu().numpy().flatten()
        
        preds_ds1_prob.extend(probs)
        trues_ds1.extend(labels)

final_preds_ds1 = (np.array(preds_ds1_prob) > optimal_ds1_threshold).astype(int)

print("\n========================================")
print("DS1 (Detection: Smelly vs Clean) TEST SET RESULTS")
print("========================================")
print(f"Target Baseline (CuBERT - Kovačević): F1 ~ 0.530")
print(f"Target Baseline (RF - Madeyski):      F1 ~ 0.520 (MCC ~ 0.510)\n")

test_f1 = f1_score(trues_ds1, final_preds_ds1)
test_mcc = matthews_corrcoef(trues_ds1, final_preds_ds1)

print(f"🔥 OUR SOTA MODEL F1-Score: {test_f1:.4f}")
print(f"🔥 OUR SOTA MODEL MCC:      {test_mcc:.4f}\n")
print(classification_report(trues_ds1, final_preds_ds1, target_names=['Clean', 'Smelly'], zero_division=0))

# ==========================================
# PHASE 5: PUBLICATION VISUALIZATIONS
# ==========================================
print("\n--- Phase 5: Generating Visualizations ---")

cm = confusion_matrix(trues_ds1, final_preds_ds1)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Clean', 'Smelly'], yticklabels=['Clean', 'Smelly'],
            annot_kws={"size": 16, "weight": "bold"})
plt.ylabel('Ground Truth', fontsize=12, fontweight='bold')
plt.xlabel('Model Prediction', fontsize=12, fontweight='bold')
plt.title(f'DS1 Detection: {SMELL_TYPE.capitalize()}', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/ds1_confusion_matrix.png", dpi=600)
plt.close()

def save_rationale_html(sample_idx, case_name):
    sample = test_dataset[sample_idx]
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    mask = sample['attention_mask'].unsqueeze(0).to(device)
    metrics = sample['metrics'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits, rationale_weights = model(input_ids, mask, metrics)
    
    prob = torch.sigmoid(logits).cpu().numpy()[0][0]
    pred_label = "Smelly" if prob > optimal_ds1_threshold else "Clean"
    true_label = "Smelly" if sample['label'].item() == 1.0 else "Clean"
    
    attentions = rationale_weights[0].cpu().numpy()
    if attentions.max() > attentions.min():
        attentions = (attentions - attentions.min()) / (attentions.max() - attentions.min())
    
    tokenizer = test_dataset.tokenizer 
    tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'])
    
    html = f"""
    <!DOCTYPE html><html><head><style>
        body {{ font-family: 'Helvetica', 'Arial', sans-serif; padding: 40px; background-color: #f4f4f4; }}
        .paper-figure {{ background: white; padding: 30px; border: 1px solid #ccc; max-width: 800px; margin: 0 auto; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        h2 {{ border-bottom: 2px solid #333; padding-bottom: 10px; margin-top: 0; }}
        .meta-info {{ display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 20px; margin-bottom: 20px; background: #eee; padding: 15px; border-radius: 5px; }}
        .meta-item strong {{ display: block; font-size: 10px; color: #555; text-transform: uppercase; letter-spacing: 1px; }}
        .meta-item span {{ font-size: 16px; font-weight: bold; }}
        .code-snippet {{ font-family: 'Consolas', 'Monaco', monospace; font-size: 13px; line-height: 1.6; background: #f9f9f9; padding: 20px; border: 1px solid #e1e1e1; overflow-x: auto; white-space: pre-wrap; }}
    </style></head><body>
    <div class="paper-figure">
        <h2>Rationale Analysis: {case_name.replace('_', ' ')}</h2>
        <div class="meta-info">
            <div class="meta-item"><strong>Prediction</strong><span style="color: {'red' if pred_label == 'Smelly' else 'green'}">{pred_label} ({(prob*100):.1f}%)</span></div>
            <div class="meta-item"><strong>Ground Truth</strong><span>{true_label}</span></div>
            <div class="meta-item"><strong>Sample ID</strong><span>{sample['sample_id']}</span></div>
        </div>
        <div class="code-snippet">
    """
    for token, score in zip(tokens, attentions):
        if token in ['<s>', '</s>', '<pad>']: continue
        text = token.replace('Ġ', ' ').replace('Ċ', '\n') 
        if score > 0.1:
            alpha = (score - 0.1) / 0.9
            html += f"<span style='background-color: rgba(255, {int(200 * (1-alpha))}, 0, 0.4); border-bottom: 2px solid rgba(255,0,0,{alpha});'>{text}</span>"
        else:
            html += f"<span>{text}</span>"
    html += "</div></div></body></html>"
    
    with open(f"{FIG_DIR}/{case_name}.html", "w", encoding='utf-8') as f:
        f.write(html)

print("Finding interesting examples for Rationale HTMLs...")
found = {'TP': False, 'FP': False, 'FN': False}
for i in range(len(test_dataset)):
    true_label = int(test_dataset[i]['label'].item())
    
    should_run = False
    if not found['TP'] and true_label == 1: should_run = True
    if not found['FP'] and true_label == 0: should_run = True
    if not found['FN'] and true_label == 1: should_run = True
    if not should_run: continue

    with torch.no_grad():
        s = test_dataset[i]
        logits, _ = model(s['input_ids'].unsqueeze(0).to(device), s['attention_mask'].unsqueeze(0).to(device), s['metrics'].unsqueeze(0).to(device))
        prob = torch.sigmoid(logits).cpu().numpy()[0][0]
        pred_label = 1 if prob > optimal_ds1_threshold else 0

    if not found['TP'] and true_label == 1 and pred_label == 1:
        save_rationale_html(i, "True_Positive_Blob")
        found['TP'] = True
    if not found['FP'] and true_label == 0 and pred_label == 1:
        save_rationale_html(i, "False_Positive_Blob")
        found['FP'] = True
    if not found['FN'] and true_label == 1 and pred_label == 0:
        save_rationale_html(i, "False_Negative_Blob")
        found['FN'] = True
    if all(found.values()): break

print("\n>>> KAGGLE PIPELINE SUCCESSFULLY COMPLETED <<<")

Using device: cuda

--- Phase 1: Data Preparation ---
Loading PMD Features...
Loading MLCQ Labels & Generating DS1 Binary Targets...


DS1 Class Distribution -> Clean: 1907, Smelly: 226
BCE Positive Weight applied: 8.44
Initializing Tokenizer...


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 14 | Loss: 0.1788 | Val F1: 0.2105 | Val MCC: 0.0993 (at Threshold 0.10)


Epoch 15:   0%|          | 0/1706 [00:00<?, ?it/s]

Epoch 15 | Loss: 0.1175 | Val F1: 0.1961 | Val MCC: 0.0902 (at Threshold 0.50)

--- Phase 4: Final Testing ---
Applying Dynamically Tuned F1 Threshold: 0.70


Testing:   0%|          | 0/214 [00:00<?, ?it/s]


DS1 (Detection: Smelly vs Clean) TEST SET RESULTS
Target Baseline (CuBERT - Kovačević): F1 ~ 0.530
Target Baseline (RF - Madeyski):      F1 ~ 0.520 (MCC ~ 0.510)

🔥 OUR SOTA MODEL F1-Score: 0.3158
🔥 OUR SOTA MODEL MCC:      0.2206

              precision    recall  f1-score   support

       Clean       0.92      0.87      0.89       191
      Smelly       0.26      0.39      0.32        23

    accuracy                           0.82       214
   macro avg       0.59      0.63      0.61       214
weighted avg       0.85      0.82      0.83       214


--- Phase 5: Generating Visualizations ---
Finding interesting examples for Rationale HTMLs...

>>> KAGGLE PIPELINE SUCCESSFULLY COMPLETED <<<
